In [1]:
# ============================================================
# Model 3 (NEW) -- Bidirectional LSTM, trained from scratch
# Paste this as a new cell/notebook. Satisfies report section
# 5.3 "RNN-Based Model" -- an actual recurrent architecture,
# unlike your current model3-deberta run (which is a second
# transformer, not an RNN).
#
# Architecture: Embedding -> 2-layer BiLSTM -> concat final
# forward+backward hidden states -> FFN head -> 1 logit,
# applied per option (same multiple-choice framing as your
# other models: 5 forward passes per example, one per option).
#
# Includes the same fixes as your BERT/DeBERTa cells:
#   - WandB key from Kaggle Secrets (never hardcoded)
#   - Stratified train/val split (comparable to your other runs)
#   - Logs train/val F1 (macro), accuracy, and val MAP@3
#   - Saves the BEST checkpoint by val F1
# ============================================================
import os
import re
from collections import Counter

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb

# ---- WandB auth (no hardcoded keys) ----
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as e:
    print(f"Could not load WANDB_API_KEY from Kaggle Secrets ({e}). Using offline mode.")
    os.environ["WANDB_MODE"] = "offline"

# ---- Load Data ----
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# ---- Config ----
MAX_LEN = 64
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
BATCH_SIZE = 32
EPOCHS = 8
LR = 1e-3
MIN_WORD_COUNT = 2
options = ['A', 'B', 'C', 'D', 'E']
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

wandb.init(project="24f3002284-t22026", name="model3-bilstm", config={
    "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM, "num_layers": NUM_LAYERS,
    "max_len": MAX_LEN, "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR
})

# ---- Tokenizer / Vocab (word-level, same convention as your scratch FFN model) ----
def tokenize(text):
    return re.findall(r"\w+", str(text).lower())

def build_vocab(df, min_count=MIN_WORD_COUNT):
    counter = Counter()
    for _, row in df.iterrows():
        counter.update(tokenize(row['prompt']))
        for opt in options:
            counter.update(tokenize(row[opt]))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, count in counter.items():
        if count >= min_count:
            vocab[word] = len(vocab)
    return vocab

full_df = pd.concat([train, test], ignore_index=True, sort=False)
vocab = build_vocab(full_df)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

import json
with open('vocab_bilstm.json', 'w') as f:
  json.dump(vocab, f)

def encode(text, max_len=MAX_LEN):
    tokens = tokenize(text)[:max_len]
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    ids += [vocab['<PAD>']] * (max_len - len(ids))
    return ids, min(len(tokens), max_len)


# ---- Dataset ----
class BiLSTMMCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids, _ = encode(row['prompt'])
        option_seqs, option_lens = [], []
        for opt in options:
            combined = str(row['prompt']) + ' ' + str(row[opt])
            ids, length = encode(combined)
            option_seqs.append(torch.tensor(ids, dtype=torch.long))
            option_lens.append(length if length > 0 else 1)  # avoid zero-length sequences
        option_seqs = torch.stack(option_seqs)          # (5, max_len)
        option_lens = torch.tensor(option_lens, dtype=torch.long)  # (5,)
        if not self.is_test:
            label = torch.tensor(label_map[row['answer']], dtype=torch.long)
            return option_seqs, option_lens, label
        return option_seqs, option_lens


def collate_fn(batch):
    if len(batch[0]) == 3:
        seqs, lens, labels = zip(*batch)
        labels = torch.stack(labels)
    else:
        seqs, lens = zip(*batch)
        labels = None
    seqs = torch.stack(seqs)   # (batch, 5, max_len)
    lens = torch.stack(lens)   # (batch, 5)
    return seqs, lens, labels


# ---- Model: Embedding -> BiLSTM -> concat final hidden states -> FFN ----
class BiLSTMMCQModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),  # *2 for bidirectional
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward_one_option(self, seq, lengths):
        # seq: (batch, max_len), lengths: (batch,)
        embedded = self.embedding(seq)  # (batch, max_len, embed_dim)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        # h_n: (num_layers * 2, batch, hidden_dim) -- take last layer's fwd+bwd states
        h_fwd = h_n[-2, :, :]  # last layer, forward direction
        h_bwd = h_n[-1, :, :]  # last layer, backward direction
        final = torch.cat([h_fwd, h_bwd], dim=1)  # (batch, hidden_dim*2)
        return self.head(final).squeeze(-1)  # (batch,)

    def forward(self, seqs, lens):
        # seqs: (batch, 5, max_len), lens: (batch, 5)
        logits = []
        for i in range(seqs.size(1)):
            logits.append(self.forward_one_option(seqs[:, i, :], lens[:, i]))
        return torch.stack(logits, dim=1)  # (batch, 5)


def map_at_3(y_true_idx, logits):
    top3 = torch.topk(logits, 3, dim=1).indices
    scores = []
    for true_idx, pred_idx in zip(y_true_idx, top3):
        pred_list = pred_idx.tolist()
        scores.append(1.0 / (pred_list.index(true_idx) + 1) if true_idx in pred_list else 0.0)
    return sum(scores) / len(scores)


# ---- Training ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_df, val_df = train_test_split(
    train, test_size=0.2, random_state=42, stratify=train['answer']
)
train_loader = DataLoader(BiLSTMMCQDataset(train_df), batch_size=BATCH_SIZE,
                           shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(BiLSTMMCQDataset(val_df), batch_size=BATCH_SIZE,
                         collate_fn=collate_fn)

model = BiLSTMMCQModel(vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

best_val_f1 = -1
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    train_true, train_pred = [], []
    for seqs, lens, labels in train_loader:
        seqs, lens, labels = seqs.to(device), lens.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(seqs, lens)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        train_true += labels.cpu().tolist()
        train_pred += logits.argmax(1).cpu().tolist()

    train_acc = accuracy_score(train_true, train_pred)
    train_f1 = f1_score(train_true, train_pred, average='macro')

    model.eval()
    val_true, val_pred, val_logits_all = [], [], []
    with torch.no_grad():
        for seqs, lens, labels in val_loader:
            seqs, lens, labels = seqs.to(device), lens.to(device), labels.to(device)
            logits = model(seqs, lens)
            val_true += labels.cpu().tolist()
            val_pred += logits.argmax(1).cpu().tolist()
            val_logits_all.append(logits.cpu())

    val_acc = accuracy_score(val_true, val_pred)
    val_f1 = f1_score(val_true, val_pred, average='macro')
    val_map3 = map_at_3(val_true, torch.cat(val_logits_all, dim=0))

    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, "
          f"Train Acc={train_acc:.4f}, Train F1={train_f1:.4f}, "
          f"Val Acc={val_acc:.4f}, Val F1={val_f1:.4f}, Val MAP@3={val_map3:.4f}")

    wandb.log({
        "epoch": epoch + 1, "loss": total_loss / len(train_loader),
        "train_acc": train_acc, "train_f1": train_f1,
        "val_acc": val_acc, "val_f1": val_f1, "val_map3": val_map3,
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'model3_bilstm_best.pth')
        wandb.save('model3_bilstm_best.pth')

wandb.summary["best_val_f1"] = best_val_f1
wandb.finish()
print("Training done!")

# ============================================================
# Inference -- loads the BEST checkpoint (by val F1).
# ============================================================
model.load_state_dict(torch.load('model3_bilstm_best.pth', map_location=device))
model.eval()
test_dataset = BiLSTMMCQDataset(test, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

all_preds = []
with torch.no_grad():
    for seqs, lens, _ in test_loader:
        seqs, lens = seqs.to(device), lens.to(device)
        logits = model(seqs, lens)
        top3 = torch.topk(logits, 3, dim=1).indices.cpu().numpy()
        all_preds.extend(top3)

predictions = [' '.join([options[i] for i in pred]) for pred in all_preds]
submission = pd.DataFrame({'ID': test['id'], 'Prediction': predictions})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print("Done!")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 24f3002284 (24f3002284-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260818_030412-1hp8pyce
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model3-bilstm
wandb: ⭐️ View project at https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026
wandb: 🚀 View run at https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026/runs/1hp8pyce


Vocab size: 2982
Using device: cuda


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 1: Loss=1.3723, Train Acc=0.4050, Train F1=0.3997, Val Acc=0.7025, Val F1=0.6992, Val MAP@3=0.8121
Epoch 2: Loss=0.5102, Train Acc=0.7981, Train F1=0.7986, Val Acc=0.9200, Val F1=0.9192, Val MAP@3=0.9550
Epoch 3: Loss=0.1813, Train Acc=0.9319, Train F1=0.9312, Val Acc=0.9725, Val F1=0.9720, Val MAP@3=0.9783
Epoch 4: Loss=0.0872, Train Acc=0.9637, Train F1=0.9639, Val Acc=0.9725, Val F1=0.9719, Val MAP@3=0.9771
Epoch 5: Loss=0.0621, Train Acc=0.9738, Train F1=0.9740, Val Acc=0.9700, Val F1=0.9706, Val MAP@3=0.9796
Epoch 6: Loss=0.0571, Train Acc=0.9744, Train F1=0.9748, Val Acc=0.9700, Val F1=0.9704, Val MAP@3=0.9800
Epoch 7: Loss=0.0630, Train Acc=0.9738, Train F1=0.9741, Val Acc=0.9725, Val F1=0.9734, Val MAP@3=0.9788


wandb: updating run metadata


Epoch 8: Loss=0.0470, Train Acc=0.9806, Train F1=0.9813, Val Acc=0.9725, Val F1=0.9725, Val MAP@3=0.9817


wandb: uploading model3_bilstm_best.pth; uploading wandb-summary.json; uploading config.yaml
wandb: uploading model3_bilstm_best.pth; uploading config.yaml
wandb: uploading model3_bilstm_best.pth
wandb: uploading history steps 6-7, summary, console lines 9-10
wandb: 
wandb: Run history:
wandb:     epoch ▁▂▃▄▅▆▇█
wandb:      loss █▃▂▁▁▁▁▁
wandb: train_acc ▁▆▇█████
wandb:  train_f1 ▁▆▇█████
wandb:   val_acc ▁▇██████
wandb:    val_f1 ▁▇██████
wandb:  val_map3 ▁▇██████
wandb: 
wandb: Run summary:
wandb: best_val_f1 0.97338
wandb:       epoch 8
wandb:        loss 0.04704
wandb:   train_acc 0.98062
wandb:    train_f1 0.98131
wandb:     val_acc 0.9725
wandb:      val_f1 0.97245
wandb:    val_map3 0.98167
wandb: 
wandb: 🚀 View run model3-bilstm at: https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026/runs/1hp8pyce
wandb: ⭐️ View project at: https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 1 other file(

Training done!
   ID Prediction
0   1      A E C
1   2      B C E
2   3      B E A
3   4      E C A
4   5      C A D
Done!
